# axonmesh: reproduce the cascade result in one notebook

The claim on the README: **half the bandwidth, 98% of the accuracy, neither
model retrained** — yolo11n answering on the device, yolo11m consulted only
for frames it is unsure about. This notebook reruns that measurement from
scratch on public weights and data, so the number is yours, not ours.

Runtime: a few minutes on a GPU runtime, ~20 minutes on CPU. Nothing here
needs a real device or a cluster — `cascade` prices the configuration
offline through ultralytics' own validator.


In [ ]:
%pip install -q git+https://github.com/dantonioluigi/axonmesh


In [ ]:
import torch
from ultralytics.data.utils import check_det_dataset

DEVICE = "0" if torch.cuda.is_available() else "cpu"
spec = check_det_dataset("coco128.yaml")  # downloads 128 COCO images (~7 MB)
IMAGES = spec["train"]
print("device:", DEVICE, "| images:", IMAGES)


## 1. Pick the routing threshold — without labels

The threshold is compared against a detector confidence score, which is not
a probability. `calibrate` runs both models over (unlabelled) frames and
asks, per frame, *would the cloud have disagreed?* — then returns the most
faithful threshold that fits a bandwidth budget.


In [ ]:
!axonmesh calibrate --edge yolo11n.pt --cloud yolo11m.pt \
    --images $IMAGES --imgsz 320 --statistic mean --max-kb 5 --device $DEVICE


Expected: the sweep chooses `--conf-high 0.60`. That is also what the
labelled mAP measurement picks independently — the cross-check is in
[docs/cascade.md](https://github.com/dantonioluigi/axonmesh/blob/main/docs/cascade.md).


## 2. Price the cascade on both axes at once

`cascade` validates three configurations with the same validator on the
same dataset: edge-only, cloud-only (every frame shipped as JPEG q50, and
the cloud scores the *decoded* image), and the cascade routing between
them. Escalated frames pay the codec they are billed for.


In [ ]:
!axonmesh cascade --edge yolo11n.pt --cloud yolo11m.pt \
    --data coco128.yaml --imgsz 320 --conf-high 0.6 --statistic mean --device $DEVICE


## What to look for

- `mean_bytes` around **5.4 KB/frame** against ~11.2 KB for always-send —
  about half the bandwidth.
- `map50_95` around **0.440** against 0.448 cloud-only — ~98% retained.
- Escalation near **47%** of frames; the other half never leave the device
  as pixels at all (11 bytes per detection).

Small caveats that matter, spelled out in
[docs/cascade.md](https://github.com/dantonioluigi/axonmesh/blob/main/docs/cascade.md):
coco128 overlaps both models' training data, so the absolute mAPs are
optimistic for *both* rows — the load-bearing result is the gap between the
curves, not either number alone. And the negative result this grew out of —
why compressing intermediate features loses to sending the frame — is in
[docs/validation.md](https://github.com/dantonioluigi/axonmesh/blob/main/docs/validation.md).
- The **compute lines**: the large model runs only on escalated frames.
  On a GPU runtime this is the in-cluster reading — the share of the
  accelerator's work the routing avoided, timed with one clock for both
  models. The share transfers between machines; the milliseconds do not.
